# Training U-Net Binary Segmentation Batubara

Notebook ini diselaraskan dengan pipeline dataset rectified dari Script 6 dan mask binary 0/1 dari Script 8. Gunakan GPU Colab: **Runtime → Change runtime type → GPU**.

Dataset yang disarankan: `dataset_unet_binary_roboflow_FIXED_01.zip` di Google Drive.


In [ ]:
"""
SCRIPT 9 — TRAINING U-NET BINARY SEGMENTATION BATUBARA (GOOGLE COLAB)
=====================================================================

Input dataset yang disarankan:
dataset_unet_binary_roboflow_FIXED_01/
    train/images/
    train/masks/
    valid/images/
    valid/masks/
    test/images/
    test/masks/

Target mask:
0 = background
1 = batubara

Output:
hasil_training_unet_colab/
    model_best.pth
    model_last.pth
    training_history.csv
    hasil_test_model_best.txt
    preview_valid/
    preview_test/
    curves_training.png

Catatan Colab:
- Upload dataset fixed 0/1 ke Google Drive, sebaiknya dalam bentuk ZIP.
- Aktifkan GPU: Runtime -> Change runtime type -> GPU.
- Jika memori GPU kurang, turunkan BATCH_SIZE menjadi 1 atau ubah IMAGE_WIDTH/HEIGHT menjadi 512x384.
"""

import os
import csv
import time
import random
import zipfile
import shutil
from glob import glob

import cv2
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt



In [ ]:
# =========================================================
# KONFIGURASI COLAB


In [ ]:
# =========================================================
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

MOUNT_DRIVE = True

# UBAH BAGIAN INI SESUAI LOKASI GOOGLE DRIVE ANDA
PROJECT_DIR = "/content/drive/MyDrive/tesis_batubara_unet" if IN_COLAB else os.path.dirname(os.path.abspath(__file__))
DATASET_ZIP = os.path.join(PROJECT_DIR, "dataset_unet_binary_roboflow_FIXED_01.zip")
DATASET_ROOT = os.path.join(PROJECT_DIR, "dataset_unet_binary_roboflow_FIXED_01")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "hasil_training_unet_colab")

# Ukuran input. Karena pipeline depth/stereomap Anda 640x480, default dibuat 640x480.
# Jika Colab GPU kehabisan memori, ubah ke 512x384 dan gunakan ukuran yang sama saat inference.
IMAGE_WIDTH = 640
IMAGE_HEIGHT = 480

# Parameter training
EPOCHS = 100
BATCH_SIZE = 2
LEARNING_RATE = 1e-4
NUM_WORKERS = 2
BASE_CHANNELS = 32

# Early stopping dan scheduler
EARLY_STOPPING_PATIENCE = 12
MIN_DELTA_IOU = 1e-4
SCHEDULER_PATIENCE = 5

# Mixed precision mempercepat training di GPU Colab
USE_AMP = True

# Threshold inference binary segmentation
THRESHOLD = 0.5

# Reproducibility
SEED = 42
SAVE_PREVIEW_COUNT = 12

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")




In [ ]:
# =========================================================
# SETUP


In [ ]:
# =========================================================
def mount_drive_if_needed():
    if IN_COLAB and MOUNT_DRIVE:
        drive.mount('/content/drive')


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True


def prepare_dataset():
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if os.path.isdir(DATASET_ROOT):
        print(f"[OK] Dataset root ditemukan: {DATASET_ROOT}")
        return DATASET_ROOT

    if os.path.exists(DATASET_ZIP):
        print(f"[INFO] Mengekstrak dataset ZIP:\n       {DATASET_ZIP}")
        with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
            z.extractall(PROJECT_DIR)

        if os.path.isdir(DATASET_ROOT):
            print(f"[OK] Dataset berhasil diekstrak: {DATASET_ROOT}")
            return DATASET_ROOT

    # Fallback: cari folder yang memiliki train/images dan train/masks
    for root, dirs, files in os.walk(PROJECT_DIR):
        if os.path.isdir(os.path.join(root, "train", "images")) and os.path.isdir(os.path.join(root, "train", "masks")):
            print(f"[OK] Dataset ditemukan otomatis: {root}")
            return root

    raise FileNotFoundError(
        "Dataset tidak ditemukan. Upload ZIP dataset_unet_binary_roboflow_FIXED_01.zip "
        "ke PROJECT_DIR atau pastikan folder dataset_unet_binary_roboflow_FIXED_01 sudah ada."
    )




In [ ]:
# =========================================================
# DATASET


In [ ]:
# =========================================================
def cari_pasangan_image_mask(root, split):
    images_dir = os.path.join(root, split, "images")
    masks_dir = os.path.join(root, split, "masks")

    if not os.path.isdir(images_dir):
        raise FileNotFoundError(f"Folder images tidak ditemukan: {images_dir}")
    if not os.path.isdir(masks_dir):
        raise FileNotFoundError(f"Folder masks tidak ditemukan: {masks_dir}")

    mask_files = sorted(glob(os.path.join(masks_dir, "*.png")))
    image_exts = [".png", ".jpg", ".jpeg", ".bmp", ".webp"]

    pairs = []
    for mask_path in mask_files:
        mask_name = os.path.basename(mask_path)
        base = os.path.splitext(mask_name)[0]

        kandidat = []
        if base.endswith("_mask"):
            base_img = base[:-5]
            for ext in image_exts:
                kandidat.append(base_img + ext)
        for ext in image_exts:
            kandidat.append(base + ext)

        image_path = None
        for k in kandidat:
            p = os.path.join(images_dir, k)
            if os.path.exists(p):
                image_path = p
                break

        if image_path is None:
            # fallback: cari file yang diawali nama base tanpa _mask
            base_no_mask = base[:-5] if base.endswith("_mask") else base
            for fname in os.listdir(images_dir):
                if fname.startswith(base_no_mask) and os.path.splitext(fname)[1].lower() in image_exts:
                    image_path = os.path.join(images_dir, fname)
                    break

        if image_path is None:
            print(f"[WARNING] Image pasangan tidak ditemukan untuk mask: {mask_name}")
            continue

        pairs.append((image_path, mask_path))

    return pairs


class CoalSegmentationDataset(Dataset):
    def __init__(self, root, split):
        self.root = root
        self.split = split
        self.pairs = cari_pasangan_image_mask(root, split)
        if len(self.pairs) == 0:
            raise RuntimeError(f"Tidak ada pasangan image-mask untuk split: {split}")
        print(f"[OK] {split}: {len(self.pairs)} pasangan image-mask")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        image_path, mask_path = self.pairs[idx]

        img = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if img is None:
            raise RuntimeError(f"Gagal membaca image: {image_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_LINEAR)
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise RuntimeError(f"Gagal membaca mask: {mask_path}")
        mask = cv2.resize(mask, (IMAGE_WIDTH, IMAGE_HEIGHT), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 0).astype(np.float32)
        mask = np.expand_dims(mask, axis=0)

        return {
            "image": torch.from_numpy(img),
            "mask": torch.from_numpy(mask),
            "image_path": image_path,
            "mask_path": mask_path,
        }




In [ ]:
# =========================================================
# MODEL U-NET


In [ ]:
# =========================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_channels, out_channels))

    def forward(self, x):
        return self.net(x)


class Up(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diff_y = x2.size(2) - x1.size(2)
        diff_x = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [diff_x // 2, diff_x - diff_x // 2, diff_y // 2, diff_y - diff_y // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, base_channels=32):
        super().__init__()
        b = base_channels
        self.inc = DoubleConv(in_channels, b)
        self.down1 = Down(b, b * 2)
        self.down2 = Down(b * 2, b * 4)
        self.down3 = Down(b * 4, b * 8)
        self.down4 = Down(b * 8, b * 16)
        self.up1 = Up(b * 16, b * 8)
        self.up2 = Up(b * 8, b * 4)
        self.up3 = Up(b * 4, b * 2)
        self.up4 = Up(b * 2, b)
        self.outc = nn.Conv2d(b, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)




In [ ]:
# =========================================================
# LOSS DAN METRIC


In [ ]:
# =========================================================
def hitung_pos_weight(dataset, max_samples=None):
    pos = 0
    neg = 0
    n = len(dataset) if max_samples is None else min(len(dataset), max_samples)
    for i in range(n):
        m = dataset[i]["mask"].numpy()
        pos += int((m > 0).sum())
        neg += int((m == 0).sum())
    if pos == 0:
        return 1.0
    weight = neg / max(pos, 1)
    return float(np.clip(weight, 1.0, 20.0))


def dice_loss_from_logits(logits, targets, smooth=1e-6):
    probs = torch.sigmoid(logits)
    probs = probs.view(probs.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    intersection = (probs * targets).sum(dim=1)
    dice = (2.0 * intersection + smooth) / (probs.sum(dim=1) + targets.sum(dim=1) + smooth)
    return 1.0 - dice.mean()


def bce_dice_loss(logits, targets, bce_loss_fn):
    bce = bce_loss_fn(logits, targets)
    dice = dice_loss_from_logits(logits, targets)
    return bce + dice


def hitung_metric(logits, targets, threshold=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    preds_flat = preds.view(preds.size(0), -1)
    targets_flat = targets.view(targets.size(0), -1)

    intersection = (preds_flat * targets_flat).sum(dim=1)
    union = preds_flat.sum(dim=1) + targets_flat.sum(dim=1) - intersection

    iou = ((intersection + eps) / (union + eps)).mean().item()
    dice = ((2 * intersection + eps) / (preds_flat.sum(dim=1) + targets_flat.sum(dim=1) + eps)).mean().item()
    pixel_acc = (preds_flat == targets_flat).float().mean().item()
    return iou, dice, pixel_acc




In [ ]:
# =========================================================
# TRAIN DAN EVALUASI


In [ ]:
# =========================================================
def train_one_epoch(model, loader, optimizer, scaler, bce_loss_fn):
    model.train()
    total_loss = total_iou = total_dice = total_acc = 0.0

    for batch in loader:
        images = batch["image"].to(DEVICE, dtype=torch.float32)
        masks = batch["mask"].to(DEVICE, dtype=torch.float32)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(USE_AMP and DEVICE.type == "cuda")):
            logits = model(images)
            loss = bce_dice_loss(logits, masks, bce_loss_fn)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        iou, dice, acc = hitung_metric(logits.detach(), masks.detach(), THRESHOLD)
        total_loss += loss.item()
        total_iou += iou
        total_dice += dice
        total_acc += acc

    n = max(len(loader), 1)
    return total_loss / n, total_iou / n, total_dice / n, total_acc / n


@torch.no_grad()
def evaluate(model, loader, bce_loss_fn):
    model.eval()
    total_loss = total_iou = total_dice = total_acc = 0.0

    for batch in loader:
        images = batch["image"].to(DEVICE, dtype=torch.float32)
        masks = batch["mask"].to(DEVICE, dtype=torch.float32)

        with torch.cuda.amp.autocast(enabled=(USE_AMP and DEVICE.type == "cuda")):
            logits = model(images)
            loss = bce_dice_loss(logits, masks, bce_loss_fn)

        iou, dice, acc = hitung_metric(logits, masks, THRESHOLD)
        total_loss += loss.item()
        total_iou += iou
        total_dice += dice
        total_acc += acc

    n = max(len(loader), 1)
    return total_loss / n, total_iou / n, total_dice / n, total_acc / n




In [ ]:
# =========================================================
# PREVIEW DAN PLOT


In [ ]:
# =========================================================
def tensor_image_to_bgr(tensor_img):
    img = tensor_img.detach().cpu().numpy()
    img = np.transpose(img, (1, 2, 0))
    img = np.clip(img * 255.0, 0, 255).astype(np.uint8)
    return cv2.cvtColor(img, cv2.COLOR_RGB2BGR)


def mask_to_visual(mask):
    mask = mask.detach().cpu().numpy()
    if mask.ndim == 3:
        mask = mask[0]
    return (mask > 0.5).astype(np.uint8) * 255


def buat_overlay_pred(img_bgr, pred_mask):
    warna = np.zeros_like(img_bgr)
    warna[pred_mask > 0] = (0, 255, 0)
    return cv2.addWeighted(img_bgr, 0.65, warna, 0.35, 0)


@torch.no_grad()
def simpan_preview(model, dataset, split_name, max_count=10):
    model.eval()
    out_dir = os.path.join(OUTPUT_DIR, f"preview_{split_name}")
    os.makedirs(out_dir, exist_ok=True)
    count = min(max_count, len(dataset))

    for i in range(count):
        sample = dataset[i]
        image = sample["image"].unsqueeze(0).to(DEVICE, dtype=torch.float32)
        gt_mask = sample["mask"]
        logits = model(image)
        prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()
        pred = (prob > THRESHOLD).astype(np.uint8) * 255

        img_bgr = tensor_image_to_bgr(sample["image"])
        gt_visual = mask_to_visual(gt_mask)
        overlay_pred = buat_overlay_pred(img_bgr, pred)
        gt_bgr = cv2.cvtColor(gt_visual, cv2.COLOR_GRAY2BGR)
        pred_bgr = cv2.cvtColor(pred, cv2.COLOR_GRAY2BGR)
        gabung = np.hstack([img_bgr, gt_bgr, pred_bgr, overlay_pred])
        cv2.putText(gabung, "IMAGE | GT MASK | PRED MASK | OVERLAY PRED", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
        out_path = os.path.join(out_dir, f"{split_name}_preview_{i+1:03d}.png")
        cv2.imwrite(out_path, gabung)

    print(f"[OK] Preview {split_name} tersimpan di: {out_dir}")


def plot_history(history_csv):
    data = np.genfromtxt(history_csv, delimiter=",", names=True)
    if data.size == 0:
        return

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(data["epoch"], data["train_loss"], label="train_loss")
    axes[0].plot(data["epoch"], data["valid_loss"], label="valid_loss")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(data["epoch"], data["train_iou"], label="train_iou")
    axes[1].plot(data["epoch"], data["valid_iou"], label="valid_iou")
    axes[1].plot(data["epoch"], data["valid_dice"], label="valid_dice")
    axes[1].set_title("Metric")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    out_path = os.path.join(OUTPUT_DIR, "curves_training.png")
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    plt.close(fig)
    print(f"[OK] Grafik training tersimpan: {out_path}")




In [ ]:
# =========================================================
# MAIN


In [ ]:
# =========================================================
def main():
    mount_drive_if_needed()
    set_seed(SEED)
    dataset_root = prepare_dataset()

    print("\n" + "=" * 78)
    print("SCRIPT 9 — TRAINING U-NET BINARY SEGMENTATION BATUBARA - COLAB")
    print("=" * 78)
    print(f"Dataset root : {dataset_root}")
    print(f"Output dir   : {OUTPUT_DIR}")
    print(f"Device       : {DEVICE}")
    print(f"Image size   : {IMAGE_WIDTH} x {IMAGE_HEIGHT}")
    print(f"Epochs max   : {EPOCHS}")
    print(f"Batch size   : {BATCH_SIZE}")
    print(f"Learning rate: {LEARNING_RATE}")
    print(f"AMP          : {USE_AMP}")
    print("=" * 78)

    train_dataset = CoalSegmentationDataset(dataset_root, "train")
    valid_dataset = CoalSegmentationDataset(dataset_root, "valid")
    test_dataset = CoalSegmentationDataset(dataset_root, "test")

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )

    pos_weight_value = hitung_pos_weight(train_dataset)
    pos_weight_tensor = torch.tensor([pos_weight_value], device=DEVICE, dtype=torch.float32)
    bce_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    print(f"[INFO] pos_weight BCE = {pos_weight_value:.4f}")

    model = UNet(in_channels=3, out_channels=1, base_channels=BASE_CHANNELS).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=SCHEDULER_PATIENCE
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and DEVICE.type == "cuda"))

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    history_path = os.path.join(OUTPUT_DIR, "training_history.csv")
    best_model_path = os.path.join(OUTPUT_DIR, "model_best.pth")
    last_model_path = os.path.join(OUTPUT_DIR, "model_last.pth")

    with open(history_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch", "train_loss", "train_iou", "train_dice", "train_pixel_acc",
            "valid_loss", "valid_iou", "valid_dice", "valid_pixel_acc",
            "learning_rate", "elapsed_sec"
        ])

    best_valid_iou = -1.0
    best_epoch = 0
    no_improve = 0

    for epoch in range(1, EPOCHS + 1):
        start = time.time()
        train_loss, train_iou, train_dice, train_acc = train_one_epoch(model, train_loader, optimizer, scaler, bce_loss_fn)
        valid_loss, valid_iou, valid_dice, valid_acc = evaluate(model, valid_loader, bce_loss_fn)
        scheduler.step(valid_loss)
        elapsed = time.time() - start
        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch [{epoch:03d}/{EPOCHS}] "
            f"Train Loss={train_loss:.4f} IoU={train_iou:.4f} Dice={train_dice:.4f} Acc={train_acc:.4f} | "
            f"Valid Loss={valid_loss:.4f} IoU={valid_iou:.4f} Dice={valid_dice:.4f} Acc={valid_acc:.4f} | "
            f"LR={current_lr:.2e} | {elapsed:.1f}s"
        )

        with open(history_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([
                epoch, train_loss, train_iou, train_dice, train_acc,
                valid_loss, valid_iou, valid_dice, valid_acc,
                current_lr, elapsed,
            ])

        if valid_iou > best_valid_iou + MIN_DELTA_IOU:
            best_valid_iou = valid_iou
            best_epoch = epoch
            no_improve = 0
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_valid_iou": best_valid_iou,
                "image_width": IMAGE_WIDTH,
                "image_height": IMAGE_HEIGHT,
                "threshold": THRESHOLD,
                "base_channels": BASE_CHANNELS,
                "pos_weight": pos_weight_value,
            }, best_model_path)
            print(f"[OK] Model terbaik disimpan: {best_model_path}")
        else:
            no_improve += 1

        if no_improve >= EARLY_STOPPING_PATIENCE:
            print(f"[INFO] Early stopping aktif. Tidak ada peningkatan valid IoU selama {EARLY_STOPPING_PATIENCE} epoch.")
            break

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_valid_iou": best_valid_iou,
        "image_width": IMAGE_WIDTH,
        "image_height": IMAGE_HEIGHT,
        "threshold": THRESHOLD,
        "base_channels": BASE_CHANNELS,
        "pos_weight": pos_weight_value,
    }, last_model_path)

    print("\n[INFO] Load model terbaik untuk evaluasi test...")
    checkpoint = torch.load(best_model_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])

    test_loss, test_iou, test_dice, test_acc = evaluate(model, test_loader, bce_loss_fn)

    print("\n" + "=" * 78)
    print("HASIL TEST MODEL TERBAIK")
    print("=" * 78)
    print(f"Best epoch     : {best_epoch}")
    print(f"Best Valid IoU : {best_valid_iou:.4f}")
    print(f"Test Loss      : {test_loss:.4f}")
    print(f"Test IoU       : {test_iou:.4f}")
    print(f"Test Dice      : {test_dice:.4f}")
    print(f"Test Pixel Acc : {test_acc:.4f}")
    print("=" * 78)

    hasil_test_path = os.path.join(OUTPUT_DIR, "hasil_test_model_best.txt")
    with open(hasil_test_path, "w", encoding="utf-8") as f:
        f.write("HASIL TEST MODEL TERBAIK\n")
        f.write("========================\n")
        f.write(f"Best epoch     : {best_epoch}\n")
        f.write(f"Best Valid IoU : {best_valid_iou:.6f}\n")
        f.write(f"Test Loss      : {test_loss:.6f}\n")
        f.write(f"Test IoU       : {test_iou:.6f}\n")
        f.write(f"Test Dice      : {test_dice:.6f}\n")
        f.write(f"Test Pixel Acc : {test_acc:.6f}\n")
        f.write(f"Image size     : {IMAGE_WIDTH}x{IMAGE_HEIGHT}\n")
        f.write(f"Batch size     : {BATCH_SIZE}\n")
        f.write(f"Learning rate  : {LEARNING_RATE}\n")
        f.write(f"Max epochs     : {EPOCHS}\n")

    simpan_preview(model, valid_dataset, "valid", SAVE_PREVIEW_COUNT)
    simpan_preview(model, test_dataset, "test", SAVE_PREVIEW_COUNT)
    plot_history(history_path)

    print("\nSELESAI TRAINING")
    print(f"Model terbaik : {best_model_path}")
    print(f"Model terakhir: {last_model_path}")
    print(f"History CSV   : {history_path}")
    print(f"Hasil test    : {hasil_test_path}")


if __name__ == "__main__":
    main()


## Catatan setelah training
- Gunakan `model_best.pth` untuk inference.
- Cek `preview_valid` dan `preview_test`; jangan hanya melihat angka IoU/Dice.
- Jika GPU out-of-memory, turunkan `BATCH_SIZE` menjadi 1 atau resolusi ke `512x384`.
